<a href="https://colab.research.google.com/github/Aljwharah-h/SDAIA-AI-Agents-System-program/blob/main/Assignment2(Redo_subagents).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install dependencies

In [33]:
%pip install -q langchain langchain_openai langchain_community

In [34]:
%pip install google-api-python-client
%pip install google-auth-httplib2
%pip install google-auth-oauthlib

In [35]:
import os

Set API Keys

In [36]:
try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

except ImportError:
    pass

In [37]:
from langchain_openai import ChatOpenAI

# https://openrouter.ai/nvidia/nemotron-3-nano-30b-a3b:free
model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

Define Tools

In [38]:
from google.oauth2.credentials import Credentials # This library is responsible for log in to your google
from google_auth_oauthlib.flow import InstalledAppFlow # This library is used for Run the Google login process (OAuth Login).
from googleapiclient.discovery import build # This library establishes a physical connection with the Google API.
import os

SCOPES = [
    "https://www.googleapis.com/auth/calendar",
    "https://www.googleapis.com/auth/gmail.send"
]

def get_google_service(service_name, version):
    creds = None

    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)

    if not creds:
        flow = InstalledAppFlow.from_client_secrets_file(
            "credentials.json", SCOPES
        )
        creds = flow.run_local_server(port=0)

        with open("token.json", "w") as token:
            token.write(creds.to_json())

    service = build(service_name, version, credentials=creds)
    return service

In [39]:
from langchain.tools import tool

@tool
def create_calendar_event(
    title: str,
    start_time: str,
    end_time: str,
    attendees: list[str],
    location: str = ""
) -> str:
    """Creates a new event on the user's Google Calendar.

    Args:
        title: The title of the calendar event.
        start_time: The start time of the event in 'YYYY-MM-DDTHH:MM:SS' format (e.g., '2023-10-27T10:00:00').
        end_time: The end time of the event in 'YYYY-MM-DDTHH:MM:SS' format (e.g., '2023-10-27T11:00:00').
        attendees: A list of email addresses of the event attendees.
        location: The physical location of the event (optional).

    Returns:
        A string containing the HTML link to the created event.
    """

    service = get_google_service("calendar", "v3")

    event = {
        "summary": title,
        "location": location,
        "start": {
            "dateTime": start_time,
            "timeZone": "UTC",
        },
        "end": {
            "dateTime": end_time,
            "timeZone": "UTC",
        },
        "attendees": [{"email": email} for email in attendees],
    }

    created_event = service.events().insert(
        calendarId="primary",
        body=event
    ).execute()

    return f"Event created: {created_event.get('htmlLink')}"

In [40]:
@tool
def send_email(
    to: list[str],
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send email via Gmail"""

    service = get_google_service("gmail", "v1")

    message = MIMEText(body)

    message["to"] = ", ".join(to)
    message["subject"] = subject

    raw = base64.urlsafe_b64encode(
        message.as_bytes()
    ).decode()

    message = {"raw": raw}

    sent = service.users().messages().send(
        userId="me",
        body=message
    ).execute()

    return f"Email sent successfully. ID: {sent['id']}"

In [41]:
@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,  # ISO format: "2024-01-15"
    duration_minutes: int
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    # Stub: In practice, this would query calendar APIs
    return ["09:00", "14:00", "16:00"]

Create a calendar agent

In [42]:
from langchain.agents import create_agent

CALENDAR_AGENT_PROMPT = (
    "You are a calendar scheduling assistant. "
    "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
    "into proper ISO datetime formats. "
    "Use get_available_time_slots to check availability when needed. "
    "Use create_calendar_event to schedule events. "
    "Always confirm what was scheduled in your final response."
)

calendar_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[
        create_calendar_event,
        get_available_time_slots
    ],
    system_prompt=CALENDAR_AGENT_PROMPT,
)

Test calendar agent

In [43]:
from langchain.messages import HumanMessage

In [44]:
query = "Schedule a team meeting next Tuesday at 2pm for 1 hour with test@gmail.com and ali@gmail.com"
result1 = calendar_agent.invoke({"messages": [HumanMessage(query)]})
for message in result1.get("messages", []):
    message.pretty_print()

FileNotFoundError: [Errno 2] No such file or directory: 'credentials.json'

Create an email agent